# Nova AI — نموذج توليد الصور الخاص بنا (Stable Diffusion، مملوك بالكامل)

**منفصل تماماً عن دفتر `merge_and_finetune.ipynb`** (فهم النص والصور) —
فهم الصور وتوليدها بنيتان مختلفتان جذرياً في الشبكات العصبية، فلا يمكن
لأي قدر من تدريب Qwen2.5-VL أن يكسبه القدرة على توليد صور. هذا نموذج
**ثانٍ ومستقل بالكامل**، لكنه بنفس مبدأ الملكية بالضبط: أوزان مفتوحة
المصدر بالكامل، نُنزّلها، نملك نسختنا الخاصة على مستودعنا، ونستضيفها
مجاناً — وليس استدعاء API مستأجَراً من أي شركة.

**لا حاجة لتشغيل هذا الدفتر أسبوعياً** كدفتر النص/الرؤية — نموذج
Stable Diffusion المُدرَّب مسبقاً يعمل بجودة عالية فور تنزيله دون أي
تدريب إضافي، ولا يوجد مصدر بيانات "يكبر تلقائياً" لتوليد الصور كما
يحدث مع محادثات نوفا النصية. شغّله **مرة واحدة** لتفعيل الميزة، ولاحقاً
مرة أو مرتين فقط إن أردت لاحقاً تخصيص أسلوب بصري مميز لنوفا عبر تدريب
LoRA خفيف (خطوة مستقبلية اختيارية، غير مبنية في هذا الدفتر بعد).

## الإعداد لمرة واحدة فقط

**1) استورد الدفتر** بنفس طريقة الدفتر الآخر تماماً: Kaggle → Create →
New Notebook → File → Import Notebook → GitHub → الصق رابط هذا الملف.
فعّل **GPU T4** من Notebook options.

**2) الأسرار (Secrets):** فقط `HF_TOKEN` و`HF_USERNAME` (نفس القيمتين
المستخدمتين في دفتر النص/الرؤية — إن كان هذا الدفتر في نفس حساب
Kaggle، الأسرار مشتركة تلقائياً ولا حاجة لإضافتها مجدداً).

**3) شغّل الخلايا بالترتيب من الأعلى للأسفل يدوياً هذه المرة** — لا
حاجة لجدولة أسبوعية (Schedule) لهذا الدفتر تحديداً، فقط شغّله عند
الحاجة.

**بعد نجاح الرفع (آخر خلية):** ضع اسم المستودع الذي تطبعه في
`HF_IMAGE_MODEL_ID` على Render (Environment Variables) ثم Manual
Deploy. عندها يعمل أمر `/صورة` في بوت نوفا فوراً.

In [ ]:
# الخلية 1 — تثبيت الأدوات
!pip install -q diffusers accelerate transformers safetensors huggingface_hub

In [ ]:
# الخلية 2 — تسجيل الدخول إلى Hugging Face (بلا أي تدخل يدوي)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("تم تسجيل الدخول إلى Hugging Face بنجاح")

In [ ]:
# الخلية 3 — الأساس المفتوح، ومستودعنا الخاص
from kaggle_secrets import UserSecretsClient

HF_USERNAME = UserSecretsClient().get_secret("HF_USERNAME")
REPO_ID = f"{HF_USERNAME}/nova-image-gen"
print("سيُرفَع نموذجنا إلى:", REPO_ID)

In [ ]:
# الخلية 4 — تحميل النموذج (بمحاولة عدة أسماء بديلة) والتحقق من عمله فعلياً
#
# owner directive 2026-09-08 (real complaint: ~15 دقيقة لصورة واحدة على
# خادم CPU بلا GPU — "هذا التأخير لا يناسبنا اطلاقاً... ابتكر شيء"):
# stabilityai/sd-turbo الآن أول مرشح، لا آخرها. هذا نموذج مُقطَّر فعلياً
# (Adversarial Diffusion Distillation) مصمَّم خصيصاً لتوليد صورة كاملة
# الجودة في 1-4 خطوات بدل 25-50 خطوة — هذا هو "الضغط" الحقيقي الذي طلبه
# المالك، وليس مجرد تقليل خطوات نموذج عادٍ تخميناً. على نفس عتاد CPU،
# تقليل الخطوات بهذه النسبة (~10-20x) يعني تخفيضاً مماثلاً تقريباً في
# زمن التوليد الفعلي. بقية القائمة تبقى كاحتياط فقط إن فشل sd-turbo
# تحديداً (مستودع نُقِل/أُزيل)، وليست الأفضل من حيث السرعة.
#
# أسماء مستودعات Stable Diffusion على Hugging Face تتغيّر بمرور الوقت
# (بالضبط كما حدث مع نموذج Gemini سابقاً في دفتر النص) — بدل الاعتماد
# على اسم واحد قد يُنقَل أو يُعاد تسميته، نجرّب عدة مستودعات معروفة
# ومستقرة تاريخياً بالترتيب حتى ينجح أحدها. إن فشلت كلها، تحقق يدوياً
# من huggingface.co/models?search=stable-diffusion لاسم حالي وأضفه
# لهذه القائمة.
import torch
from diffusers import AutoPipelineForText2Image

_CANDIDATE_IMAGE_MODELS = [
    "stabilityai/sd-turbo",
    "stabilityai/stable-diffusion-2-1",
    "stabilityai/stable-diffusion-2-1-base",
    "runwayml/stable-diffusion-v1-5",
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "CompVis/stable-diffusion-v1-4",
]

# دليل حقيقي سابق (نفس الجلسة): "kaggle kernels push" عبر الـAPI يعيد
# ضبط نوع المعالج إلى P100 الافتراضي في كل مرة، وP100 (sm_60) غير
# متوافق مع PyTorch الحديث (يحتاج sm_70+). سبق إصلاح هذا يدوياً مرة
# لدفتر الفيديو، لكن هذا الدفتر يُدفع تلقائياً بلا إنسان يراقبه، فنفس
# الخلل يتكرر بصمت ويوقف التنفيذ هنا قبل وصوله لخليتَي الرفع الفعلي
# (5ب/9ب) — رغم أن الـworkflow يُبلّغ "نجاح" (نجح في الدفع فقط، لا في
# التنفيذ). اكتشاف فعلي لتوفر CUDA بدل افتراضه، مع نزول آمن إلى CPU
# (float16 بلا CUDA بطيء/غير مدعوم على بعض العتاد؛ bfloat16 آمن ومدعوم
# على CPU) بدل الاعتماد على ضبط يدوي متكرر لا يحدث فعلياً على دفتر
# مؤتمت بالكامل.
_device = "cuda" if torch.cuda.is_available() else "cpu"
_load_dtype = torch.float16 if _device == "cuda" else torch.bfloat16
print(f"الجهاز المستخدم فعلياً لهذا التشغيل: {_device}")

pipe = None
BASE_IMAGE_MODEL_ID = None
for _candidate in _CANDIDATE_IMAGE_MODELS:
    try:
        print("تجربة:", _candidate, "...")
        pipe = AutoPipelineForText2Image.from_pretrained(_candidate, torch_dtype=_load_dtype)
        BASE_IMAGE_MODEL_ID = _candidate
        print("نجح التحميل من:", _candidate)
        break
    except Exception as e:
        print("فشل", _candidate, "-", type(e).__name__)

if pipe is None:
    raise RuntimeError(
        "فشلت كل الأسماء المرشّحة. تحقق يدوياً من huggingface.co/models?search=stable-diffusion "
        "عن اسم مستودع حالي وأضفه إلى _CANDIDATE_IMAGE_MODELS أعلاه ثم أعد تشغيل هذه الخلية."
    )

pipe = pipe.to(_device)

# sd-turbo موثَّق رسمياً بهذين الإعدادين تحديداً (num_inference_steps
# قليل جداً، guidance_scale=0.0) — استخدام إعدادات نموذج عادٍ (25 خطوة)
# معه لن يُكسِر شيئاً لكنه يهدر وقت الاختبار بلا داعٍ ولا يختبر
# الإعدادات الحقيقية التي سيستخدمها خادم التشغيل (راجع
# ai-system/modelscope-studio/app.py's generate_image).
_is_turbo = "turbo" in BASE_IMAGE_MODEL_ID.lower()
_test_steps = 2 if _is_turbo else 25
_test_guidance = 0.0 if _is_turbo else 7.5
test_image = pipe(
    "a friendly cartoon robot mascot, simple flat design",
    num_inference_steps=_test_steps, guidance_scale=_test_guidance,
).images[0]
test_image.save("test_output.png")
print("تم توليد صورة تجريبية بنجاح باستخدام", BASE_IMAGE_MODEL_ID, "— تحقق من test_output.png في ملفات الجلسة (Output) للتأكد بصرياً.")


In [ ]:
# الخلية 5 — رفع النموذج إلى مستودعنا الخاص على Hugging Face Hub
#
# نسخة كاملة من أوزان Stable Diffusion المفتوحة (BASE_IMAGE_MODEL_ID
# الذي نجح تحميله في الخلية السابقة)، منسوخة بالكامل إلى مستودعنا
# نحن — نملك نسختنا الآن بشكل كامل ومستقل عن أي تغيير مستقبلي على
# المستودع الأصلي، وجاهزة لاستضافة HF Inference المجانية بنفس طريقة
# نموذج النص/الرؤية بالضبط.
pipe.save_pretrained("./nova-image-gen-local")

# owner directive 2026-09-08: خادم التشغيل (app.py's generate_image)
# يحتاج معرفة أي مرشّح نجح فعلياً ليستخدم الإعدادات الصحيحة له (sd-turbo
# يحتاج خطوات قليلة جداً وguidance_scale=0.0؛ نموذج عادي يحتاج المخطِّط
# الأسرع DPMSolverMultistepScheduler وخطوات أكثر) — بلا هذا الملف، خادم
# التشغيل لا يملك أي طريقة ليعرف الفرق. ملف بسيط، بلا بيانات ثقيلة.
import json as _json

with open("./nova-image-gen-local/nova_gen_config.json", "w") as f:
    _json.dump({"base_model_id": BASE_IMAGE_MODEL_ID}, f)

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path="./nova-image-gen-local", repo_id=REPO_ID)
print(f"تم الرفع: https://huggingface.co/{REPO_ID}")
print("ضع هذا في HF_IMAGE_MODEL_ID داخل ai-system/.env أو Render:", REPO_ID)


In [ ]:
# الخلية 5ب — رفع نفس نموذج الصورة أيضاً إلى ModelScope (مصدر التشغيل الفعلي)
#
# owner directive 2026-09-08/09 ("قرر واعمل"): جذر فشل توليد الصور
# الحقيقي، المكتشف من سجلات ModelScope نفسها (RuntimeError داخل
# _get_image_pipe لأن _image_model_dir كان None)، بقي قائماً حتى بعد
# إصلاح إعادة المحاولة (retry-with-backoff) — أعاد المالك النشر واختبر
# ثلاث مرات، بلا أي تغيير. هذا ينفي سبباً عابراً (كخلل شبكة مؤقت) ويشير
# إلى مشكلة بنيوية: بنية ModelScope الأساسية صينية فعلياً (منطقة
# cn-zhangjiakou، ومرآة mirrors.aliyun.com لـpip — كلاهما ظهر مباشرة في
# سجلات بناء هذا الاستوديو نفسها)، وهذا نمط شبكي معروف لا يستطيع غالباً
# الوصول إلى huggingface.co إطلاقاً — رغم تأكيد المالك حياً (لقطة شاشة)
# أن مستودع Hugging Face نفسه عام وصالح وفيه الملفات. النموذج الرئيسي
# (نص/رؤية) لم يفشل ولو مرة لأنه يُنزَّل من مستودع ModelScope نفسها
# (modelscope.snapshot_download) لا من Hugging Face — هذا نفس الحل هنا:
# نرفع نسخة إضافية من نموذج الصورة إلى ModelScope مباشرة، فيقرأها
# app.py عبر نفس المسار الموثوق أصلاً بدل huggingface_hub.
!pip install -q modelscope

from kaggle_secrets import UserSecretsClient
from modelscope.hub.api import HubApi
from modelscope.hub.constants import ModelVisibility

_ms_secrets = UserSecretsClient()
MODELSCOPE_TOKEN = _ms_secrets.get_secret("MODELSCOPE_TOKEN")
MODELSCOPE_USERNAME = _ms_secrets.get_secret("MODELSCOPE_USERNAME")
MODELSCOPE_IMAGE_REPO = f"{MODELSCOPE_USERNAME}/nova-image-gen"

# push_model يتطلب configuration.json داخل مجلد الرفع (نفس شرط رفع
# GGUF في merge_and_finetune.ipynb) — diffusers لا يكتب ملفاً بهذا
# الاسم تحديداً، فنضيفه فقط إن لم يكن موجوداً أصلاً.
import json
import os

if not os.path.exists("./nova-image-gen-local/configuration.json"):
    with open("./nova-image-gen-local/configuration.json", "w") as f:
        json.dump({"framework": "pytorch", "task": "text-to-image-synthesis"}, f)

_api = HubApi()
_api.login(MODELSCOPE_TOKEN)
_api.push_model(
    model_id=MODELSCOPE_IMAGE_REPO,
    model_dir="./nova-image-gen-local",
    visibility=ModelVisibility.PUBLIC,
    license="apache-2.0",
)
print(f"تم رفع نموذج الصورة أيضاً إلى: https://modelscope.cn/models/{MODELSCOPE_IMAGE_REPO}")
print("app.py على خادم ModelScope أصبح يقرأ هذا تلقائياً (_IMAGE_MODEL_ID = \"novaai2026/nova-image-gen\") — لا حاجة لتعديل يدوي إن كان الاسم مطابقاً.")


---
## نموذج توليد الفيديو الخاص بنا (CogVideoX-2B، مملوك بالكامل)

**owner spec 2026-09-08:** نفس المبدأ بالضبط — لا نستأجر توليد فيديو من
أي شركة، بل ننزّل نموذجاً مفتوح الأوزان بالكامل ونملك نسختنا الخاصة.

**لماذا CogVideoX-2B تحديداً؟** بحثت هذا حياً (وليس تخميناً): إصداره
2B مرخّص Apache 2.0 (استخدام تجاري حر بالكامل) ومؤكَّد أنه يعمل على
بطاقات 16GB — بالضبط ما توفره Kaggle's T4 مجاناً. توليد الفيديو أثقل
بكثير من توليد الصور (كل فيديو هو عشرات الإطارات، لا إطار واحد)، لذا
الإصدار الأخف (2B) هو الخيار الواقعي الوحيد على عتاد مجاني، وليس
إصداره الأكبر (5B) الذي يحتاج ذاكرة أكبر مما نملك هنا.

**بديل احتياطي أخف حتى من ذلك:** `damo-vilab/text-to-video-ms-1.7b`
(نموذج ModelScope نفسه — نستضيف عليه أصلاً خادم النص/الرؤية الحي)،
أصغر وأضمن نجاحاً إن فشل تحميل CogVideoX-2B لأي سبب (مساحة، ذاكرة،
تغيّر اسم المستودع بمرور الوقت كما حدث سابقاً مع Stable Diffusion
أعلاه وGemini في الدفتر الآخر).

**صادق حول الجودة والسرعة:** فيديو 2B على معالج رسومي مجاني لن ينافس
منتجات تجارية مدفوعة (Sora، Veo) — هذا نموذج مفتوح صغير نسبياً، ناتجه
قصير (بضع ثوانٍ) ومتواضع الدقة مقارنة بها. الهدف هنا: ملكية حقيقية
وميزة تعمل فعلياً بتكلفة صفر، وليس منافسة أفضل الحلول التجارية.

**نفس أسلوب تشغيل الصور تماماً: مرة واحدة يدوياً، ليس أسبوعياً.**

In [ ]:
# الخلية 6 — تثبيت أدوات الفيديو الإضافية
#
# diffusers الحديثة تدعم CogVideoXPipeline (لم تكن مثبَّتة بعد لخلية
# الصور أعلاه بإصدار حديث بما يكفي دائماً) — نحدّثها صراحة هنا.
# imageio[ffmpeg] هو ما يحوّل الإطارات الناتجة إلى ملف mp4 فعلي عبر
# diffusers.utils.export_to_video.
!pip install -q -U diffusers transformers accelerate
!pip install -q "imageio[ffmpeg]"

In [ ]:
# الخلية 7 — الأساس المفتوح لتوليد الفيديو، ومستودعنا الخاص
from kaggle_secrets import UserSecretsClient

VIDEO_REPO_ID = f"{HF_USERNAME}/nova-video-gen"
print("سيُرفَع نموذج الفيديو إلى:", VIDEO_REPO_ID)

In [ ]:
# الخلية 8 — تحميل نموذج الفيديو (بمحاولة مرشَّحين مختلفي البنية) والتحقق الفعلي بتوليد فيديو حقيقي
#
# على عكس خلية الصور أعلاه (كل مرشحيها من عائلة Stable Diffusion،
# فتشترك في نفس فئة الأنبوب AutoPipelineForText2Image)، مرشَّحا الفيديو
# هنا بنيتان مختلفتان فعلياً (CogVideoX له فئة أنبوب خاصة به)، فكل
# مرشح يحمل فئته وطريقة استدعائه الخاصة بدل حلقة موحّدة واحدة.
#
# enable_model_cpu_offload/enable_vae_slicing/enable_vae_tiling: هذه
# ليست تحسينات اختيارية هنا بل ما يجعل نموذجاً بحجم CogVideoX-2B يعمل
# أصلاً ضمن 16GB بدل نفاد الذاكرة (OOM) — موثَّقة على صفحة النموذج
# الرسمية لتشغيله على عتاد بذاكرة محدودة.
import gc

import torch
from diffusers.utils import export_to_video

# owner report, 2026-09-09 (real Kaggle evidence): the kernel died with
# "appears to have died, restarting automatically" right after this
# cell's own test generation succeeded (confirmed by its own printed
# "نجح التحميل"/"تم توليد فيديو تجريبي بنجاح" lines still visible in
# the crashed output) — the crash happened in the NEXT cell's
# save_pretrained call. Root cause, not guessed: cell 4/5 above never
# released the Stable Diffusion `pipe` object (moved to .to("cuda")) —
# it's still sitting in memory when this cell loads a second, much
# heavier model (CogVideoX-2B) on top of it, so both models' weights
# coexist until something finally tips memory over the edge. Free the
# image-generation pipe here since this is a genuinely separate phase
# that no longer needs it.
if "pipe" in dir():
    del pipe
gc.collect()
torch.cuda.empty_cache()

video_pipe = None
BASE_VIDEO_MODEL_ID = None
VIDEO_KIND = None  # "cogvideox" | "text2video_ms" — تحدد طريقة الاستدعاء لاحقاً


def _try_cogvideox():
    from diffusers import CogVideoXPipeline

    pipe = CogVideoXPipeline.from_pretrained("THUDM/CogVideoX-2b", torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()
    return pipe, "THUDM/CogVideoX-2b", "cogvideox"


def _try_text2video_ms():
    from diffusers import DiffusionPipeline

    pipe = DiffusionPipeline.from_pretrained(
        "damo-vilab/text-to-video-ms-1.7b", torch_dtype=torch.float16, variant="fp16"
    )
    pipe.enable_model_cpu_offload()
    return pipe, "damo-vilab/text-to-video-ms-1.7b", "text2video_ms"


for _attempt in (_try_cogvideox, _try_text2video_ms):
    try:
        print("تجربة:", _attempt.__name__, "...")
        video_pipe, BASE_VIDEO_MODEL_ID, VIDEO_KIND = _attempt()
        print("نجح التحميل من:", BASE_VIDEO_MODEL_ID)
        break
    except Exception as e:
        print("فشل", _attempt.__name__, "-", type(e).__name__, "-", str(e)[:200])

if video_pipe is None:
    raise RuntimeError(
        "فشل تحميل كلا مرشَّحَي الفيديو. تحقق يدوياً من huggingface.co/models?pipeline_tag=text-to-video "
        "عن اسم مستودع حالي، وأضف دالة محاولة جديدة أعلاه على نمط الدالتين الموجودتين."
    )

_test_prompt = "a friendly cartoon robot mascot waving hello, simple flat design"
if VIDEO_KIND == "cogvideox":
    _frames = video_pipe(
        prompt=_test_prompt, num_videos_per_prompt=1, num_inference_steps=50, num_frames=49, guidance_scale=6
    ).frames[0]
    export_to_video(_frames, "test_output_video.mp4", fps=8)
else:
    _frames = video_pipe(prompt=_test_prompt, num_inference_steps=25, num_frames=16).frames[0]
    export_to_video(_frames, "test_output_video.mp4")

print("تم توليد فيديو تجريبي بنجاح باستخدام", BASE_VIDEO_MODEL_ID, "— تحقق من test_output_video.mp4 في ملفات الجلسة (Output) للتأكد بصرياً.")

In [ ]:
# الخلية 9 — رفع نموذج الفيديو إلى مستودعنا الخاص على Hugging Face Hub
#
# owner report, 2026-09-09 (real Kaggle evidence, TWICE now — the first
# cleanup attempt below did not fix it): the kernel died with an
# OOM-shaped crash right at save_pretrained's "Writing model shards"
# step, right after cell 8's own test generation succeeded. Root cause,
# not guessed: CogVideoX-2b's full footprint is ~13.8GB (seen directly
# in cell 8's own download log) — right at Kaggle's free-tier RAM
# ceiling — and pipe.save_pretrained() materializes ALL sub-models
# (transformer + T5 text encoder + VAE) without freeing memory between
# them, so peak RAM briefly needs close to the model's FULL size on top
# of whatever enable_model_cpu_offload() had already staged in system
# RAM from the test generation. Fix: do what save_pretrained does
# internally, but one component at a time, explicitly moved to CPU and
# freed (del + gc.collect + empty_cache) before the next — peak RAM now
# only needs to fit the single largest component (the transformer),
# not the whole pipeline at once. save_config() first (cheap metadata
# only, writes model_index.json) so the serving side can still
# reconstruct the pipeline correctly even though the live pipe object's
# component attributes get deleted along the way.
import gc

if "_frames" in dir():
    del _frames
gc.collect()
torch.cuda.empty_cache()

_video_local_dir = "./nova-video-gen-local"
video_pipe.save_config(_video_local_dir)

for _name, _component in list(video_pipe.components.items()):
    if hasattr(_component, "save_pretrained"):
        print("حفظ:", _name, "...")
        if hasattr(_component, "to"):
            _component.to("cpu")
        _component.save_pretrained(f"{_video_local_dir}/{_name}")
        delattr(video_pipe, _name)
        del _component
        gc.collect()
        torch.cuda.empty_cache()
        print("تم حفظ:", _name)

del video_pipe
gc.collect()
torch.cuda.empty_cache()

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(VIDEO_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=_video_local_dir, repo_id=VIDEO_REPO_ID)
print(f"تم الرفع: https://huggingface.co/{VIDEO_REPO_ID}")
print("النوع (لضبط HF_VIDEO_MODEL_KIND على Render):", VIDEO_KIND)
print("ضع هذا في HF_VIDEO_MODEL_ID داخل ai-system/.env أو Render:", VIDEO_REPO_ID)


In [ ]:
# الخلية 9ب — رفع نفس نموذج الفيديو أيضاً إلى ModelScope (مصدر التشغيل الفعلي)
#
# نفس السبب والحل بالضبط الموضَّحين في خلية رفع نموذج الصورة أعلاه
# (مشكلة شبكة بنيوية بين خوادم ModelScope الصينية وhuggingface.co، لا
# مشكلة صلاحيات أو مستودع) — نرفع نسخة إضافية من نموذج الفيديو مباشرة
# إلى ModelScope، فيقرأها app.py عبر نفس المسار الموثوق أصلاً.
import json
import os

!pip install -q modelscope

from kaggle_secrets import UserSecretsClient
from modelscope.hub.api import HubApi
from modelscope.hub.constants import ModelVisibility

_ms_secrets = UserSecretsClient()
MODELSCOPE_TOKEN = _ms_secrets.get_secret("MODELSCOPE_TOKEN")
MODELSCOPE_USERNAME = _ms_secrets.get_secret("MODELSCOPE_USERNAME")
MODELSCOPE_VIDEO_REPO = f"{MODELSCOPE_USERNAME}/nova-video-gen"

if not os.path.exists("./nova-video-gen-local/configuration.json"):
    with open("./nova-video-gen-local/configuration.json", "w") as f:
        json.dump({"framework": "pytorch", "task": "text-to-video-synthesis"}, f)

_api = HubApi()
_api.login(MODELSCOPE_TOKEN)
_api.push_model(
    model_id=MODELSCOPE_VIDEO_REPO,
    model_dir="./nova-video-gen-local",
    visibility=ModelVisibility.PUBLIC,
    license="apache-2.0",
)
print(f"تم رفع نموذج الفيديو أيضاً إلى: https://modelscope.cn/models/{MODELSCOPE_VIDEO_REPO}")
print("app.py على خادم ModelScope أصبح يقرأ هذا تلقائياً (_VIDEO_MODEL_ID = \"novaai2026/nova-video-gen\") — لا حاجة لتعديل يدوي إن كان الاسم مطابقاً.")
